# M4: Grid Capacity — Endesa (e-distribución)
## IE Sustainability Datathon March 2026, Iberdrola Challenge

**Objective:** Load, clean, and explore the Endesa e-distribución node-level access capacity dataset. Assign a `grid_status` classification (Sufficient / Moderate / Congested) to each substation and export a model-ready CSV for use by the network optimisation notebook.

**Data source:** Endesa e-distribución — Historical access capacity documents  
→ File: `data/external/grid_capacity_endesa/2026_03_04_R1299_demanda.csv`  
→ Coverage: Andalucía, Extremadura, Cataluña, Canarias, Baleares, parts of Aragón and Castilla-La Mancha  
→ Format: Semicolon-delimited CSV + XLSX (monthly updates)

**Role in pipeline:**  
This notebook produces `grid_status` thresholds and a cleaned substation dataset that M6 (network optimisation) uses to classify every proposed charging location. This directly determines the values in **File 2** (`grid_status` column) and **File 3** (friction points log) — both mandatory deliverables.

**Depends on:** None (standalone — uses raw Endesa file directly)  
**Output:** `data/interim/m4_endesa_grid_capacity.csv`

## 0. Setup: Dependencies & Paths

In [ ]:
# !pip install pandas numpy geopandas matplotlib scipy -q

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
CSV_PATH  = '../data/external/grid_capacity_endesa/2026_03_04_R1299_demanda.csv'
XLSX_PATH = '../data/external/grid_capacity_endesa/2026_03_04_R1299_demanda.xlsx'
RTIG_PATH = '../data/raw/road_routes_spain/carreteras_RTIG.geojson'
OUT_PATH  = '../data/interim/m4_endesa_grid_capacity.csv'

os.makedirs('../data/interim', exist_ok=True)

print('Libraries loaded.')
print(f'Source file: {CSV_PATH}')

## 1. Load Raw Data

Endesa publishes monthly access capacity documents in both CSV and XLSX. We use the CSV as the primary source and fall back to XLSX if needed. The file uses semicolon delimiters and may use a comma as the decimal separator (European locale).

**Key fields expected:**
- Node / substation identifier
- Municipality and province
- Geographic coordinates (latitude, longitude)
- Available capacity in MW (`Capacidad disponible`)
- Voltage level (kV)

> **Note on column names:** Endesa field names may vary slightly between monthly releases. The cell below prints the raw column list so you can confirm the exact names and adjust the mapping dictionary if needed.

In [ ]:
# ── Try CSV first, fall back to XLSX ─────────────────────────────────────────
if os.path.exists(CSV_PATH):
    # Try common Endesa CSV encodings and separators
    for enc in ['utf-8', 'latin1', 'cp1252']:
        try:
            df_raw = pd.read_csv(
                CSV_PATH,
                sep=';',
                encoding=enc,
                decimal=',',   # European decimal convention
                low_memory=False
            )
            print(f'Loaded CSV with encoding={enc}')
            break
        except Exception as e:
            print(f'Encoding {enc} failed: {e}')
elif os.path.exists(XLSX_PATH):
    df_raw = pd.read_excel(XLSX_PATH)
    print('Loaded XLSX (CSV not found)')
else:
    raise FileNotFoundError(
        f'Neither CSV nor XLSX found.\n'
        f'Expected: {CSV_PATH}\n'
        f'Download from: https://www.edistribucion.com/es/red-electrica/acceso-red/capacidad-acceso.html'
    )

print(f'\nRaw shape  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'\nColumn names (raw):')
for i, col in enumerate(df_raw.columns):
    print(f'  [{i:02d}] {col}')
print(f'\nFirst 3 rows:')
print(df_raw.head(3).to_string())

## 2. Column Mapping & Standardisation

Endesa column names are in Spanish and vary across releases. We map them to standardised internal names here. 

**Adjust the `COLUMN_MAP` dictionary below** if the printed column names above differ from the expected values. The comments show common alternative names seen in past releases.

In [ ]:
# ── Column mapping: Endesa name → internal name ───────────────────────────────
# Adjust keys to match the column names printed above if they differ.
COLUMN_MAP = {
    # Node identifier
    'Nudo':                    'node_id',       # alt: 'Código nudo', 'ID_Nudo'
    # Substation / node name
    'Nombre':                  'node_name',     # alt: 'Denominación', 'Subestación'
    # Municipality
    'Municipio':               'municipality',  # alt: 'Municipio'
    # Province
    'Provincia':               'province',      # alt: 'Provincia'
    # Coordinates
    'Latitud':                 'latitude',      # alt: 'LAT', 'lat'
    'Longitud':                'longitude',     # alt: 'LON', 'lon'
    # Available capacity (MW) — the key field for grid_status
    'Capacidad disponible':    'available_mw',  # alt: 'Cap. disponible (MW)', 'Capacidad_disponible_MW'
    # Voltage level (kV)
    'Tensión':                 'voltage_kv',    # alt: 'Nivel de tensión', 'kV'
    # Access type
    'Tipo de acceso':          'access_type',   # alt: 'Tipo acceso'
}

# Keep only columns that exist in this file release
cols_present = {k: v for k, v in COLUMN_MAP.items() if k in df_raw.columns}
cols_missing  = [k for k in COLUMN_MAP if k not in df_raw.columns]

df = df_raw[list(cols_present.keys())].rename(columns=cols_present).copy()

print(f'Columns mapped    : {len(cols_present)}')
if cols_missing:
    print(f'Columns not found : {cols_missing}')
    print('  → Check column names in cell above and update COLUMN_MAP accordingly.')

print(f'\nStandardised columns: {df.columns.tolist()}')
print(f'Shape: {df.shape}')

## 3. Clean & Validate

Key cleaning steps:
1. Parse `available_mw` to float (handles commas as decimal separators)
2. Validate coordinates within Spain's bounding box
3. Drop rows with missing coordinates or capacity — these cannot be spatially matched
4. Handle negative capacity values: Endesa uses `0` or negative values for nodes with no remaining capacity (i.e., fully congested)

In [ ]:
n_raw = len(df)

# ── Parse available_mw to float ───────────────────────────────────────────────
if 'available_mw' in df.columns:
    df['available_mw'] = (
        df['available_mw']
        .astype(str)
        .str.replace(',', '.', regex=False)
        .str.strip()
        .replace({'': np.nan, 'nan': np.nan, '-': np.nan})
    )
    df['available_mw'] = pd.to_numeric(df['available_mw'], errors='coerce')

# ── Parse coordinates ──────────────────────────────────────────────────────────
for col in ['latitude', 'longitude']:
    if col in df.columns:
        df[col] = (
            df[col].astype(str)
            .str.replace(',', '.', regex=False)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ── Drop rows with no coordinates or no capacity ──────────────────────────────
df = df.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
n_no_coords = n_raw - len(df)

# ── Validate Spain bounding box (includes Canaries + Balearics) ───────────────
mask_spain = (
    df['longitude'].between(-20, 5) &
    df['latitude'].between(27, 45)
)
n_outside = (~mask_spain).sum()
df = df[mask_spain].reset_index(drop=True)

# ── Clip negative capacity to 0 (fully congested nodes) ──────────────────────
if 'available_mw' in df.columns:
    n_negative = (df['available_mw'] < 0).sum()
    df['available_mw'] = df['available_mw'].clip(lower=0)
else:
    n_negative = 0

print('CLEANING SUMMARY')
print('-' * 40)
print(f'Raw rows              : {n_raw:>6,}')
print(f'Dropped (no coords)   : {n_no_coords:>6,}')
print(f'Dropped (outside ESP) : {n_outside:>6,}')
print(f'Negative MW clipped   : {n_negative:>6,}  (set to 0)')
print(f'Final clean rows      : {len(df):>6,}')
print()
print('Capacity (available_mw) statistics:')
if 'available_mw' in df.columns:
    print(df['available_mw'].describe().round(2).to_string())

## 4. Exploratory Data Analysis

Four views of the Endesa network:
1. **Capacity distribution** — histogram of available MW across all nodes
2. **By province** — which regions have the most constrained grid
3. **Voltage level breakdown** — mix of HV/MV nodes and their capacity profiles
4. **Geographic map** — spatial distribution of nodes coloured by available capacity

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle(
    'Endesa e-distribución — Grid Access Capacity EDA\n'
    '(IE Datathon 2026 / Iberdrola Challenge)',
    fontsize=13, fontweight='bold'
)

# ── 1. Capacity distribution ──────────────────────────────────────────────────
ax = axes[0, 0]
if 'available_mw' in df.columns:
    cap = df['available_mw'].dropna()
    cap_capped = cap.clip(upper=cap.quantile(0.99))  # cap outliers for readability
    ax.hist(cap_capped, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.axvline(cap.median(), color='darkorange', linestyle='--', linewidth=1.5,
               label=f'Median: {cap.median():.1f} MW')
    ax.axvline(cap.mean(), color='crimson', linestyle=':', linewidth=1.5,
               label=f'Mean: {cap.mean():.1f} MW')
    ax.set_title('Distribution of Available Capacity (MW)')
    ax.set_xlabel('Available Capacity (MW)')
    ax.set_ylabel('Node count')
    ax.legend(fontsize=9)
    ax.text(0.97, 0.95, f'n = {len(cap):,} nodes', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color='grey')

# ── 2. Top 15 provinces by node count ────────────────────────────────────────
ax = axes[0, 1]
if 'province' in df.columns:
    prov = df.groupby('province').agg(
        nodes=('node_id' if 'node_id' in df.columns else 'latitude', 'count'),
        mean_mw=('available_mw', 'mean') if 'available_mw' in df.columns else ('latitude', 'count')
    ).sort_values('nodes', ascending=True).tail(15)
    colors_bar = ['#d73027' if m < 5 else '#fee08b' if m < 20 else '#1a9850'
                  for m in prov['mean_mw']]
    bars = ax.barh(prov.index, prov['nodes'], color=colors_bar)
    ax.set_title('Top 15 Provinces by Node Count\n(colour = mean available MW: red<5, yellow<20, green≥20)')
    ax.set_xlabel('Number of nodes')
    for bar, (_, row) in zip(bars, prov.iterrows()):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f"{row['mean_mw']:.0f} MW", va='center', fontsize=7.5)

# ── 3. Capacity by voltage level ──────────────────────────────────────────────
ax = axes[1, 0]
if 'voltage_kv' in df.columns and 'available_mw' in df.columns:
    vlt = df.groupby('voltage_kv')['available_mw'].agg(['median', 'count'])
    vlt = vlt.sort_values('median', ascending=False).head(10)
    ax.bar(vlt.index.astype(str), vlt['median'], color='teal', alpha=0.8)
    ax.set_title('Median Available Capacity by Voltage Level (kV)')
    ax.set_xlabel('Voltage level (kV)')
    ax.set_ylabel('Median available capacity (MW)')
    ax.tick_params(axis='x', rotation=45)
    for i, (idx, row) in enumerate(vlt.iterrows()):
        ax.text(i, row['median'] + 0.2, f"n={int(row['count'])}",
                ha='center', fontsize=7.5, color='grey')
else:
    ax.text(0.5, 0.5, 'voltage_kv column not found\nUpdate COLUMN_MAP',
            ha='center', va='center', transform=ax.transAxes, color='grey')
    ax.set_title('Capacity by Voltage Level (data not available)')

# ── 4. Geographic map ─────────────────────────────────────────────────────────
ax = axes[1, 1]
if 'available_mw' in df.columns:
    scatter = ax.scatter(
        df['longitude'], df['latitude'],
        c=df['available_mw'].clip(upper=df['available_mw'].quantile(0.95)),
        cmap='RdYlGn', s=8, alpha=0.7
    )
    plt.colorbar(scatter, ax=ax, label='Available capacity (MW)', shrink=0.8)
    ax.set_title('Endesa Grid Nodes — Available Capacity\n(green=high, red=low/congested)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_xlim(-10, 5)
    ax.set_ylim(27, 45)

plt.tight_layout()
plt.savefig('../data/interim/m4_endesa_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA charts saved.')

### Insight — Capacity Distribution & Geographic Patterns

The EDA above reveals the key patterns for grid_status classification:

- The capacity distribution is **right-skewed**: most nodes have low-to-moderate available MW, with a long tail of high-capacity nodes near major substations.
- Provinces in **Andalucía and Cataluña** tend to have higher node counts but variable capacity — these are critical zones for the A-4 and A-7 corridors.
- **Zero-capacity nodes** (clipped from negative) represent fully congested points and will be classified as `Congested` regardless of threshold.
- Higher voltage nodes (66 kV, 132 kV) show substantially higher median capacity — these are the preferred anchoring points for HPC charger installations.

## 5. Grid Status Classification

### Threshold Definition & Justification

The datathon mandates that `grid_status` be assigned based on `Capacidad disponible (MW)` with thresholds documented and justified in the Analytical Report. We adopt the following thresholds, derived from HPC charger power requirements:

| Status | Threshold | Reasoning |
|---|---|---|
| **Sufficient** | ≥ 10 MW | Comfortably supports 60+ chargers × 150 kW = 9 MW. Node can absorb a full charging hub without reinforcement. |
| **Moderate** | 1.5 – 10 MW | Supports 10–66 chargers. Partial deployment viable; grid reinforcement needed for full-scale hub. |
| **Congested** | < 1.5 MW | Supports fewer than 10 chargers at 150 kW. Grid reinforcement is a prerequisite for any meaningful deployment. |

**Basis for thresholds:**
- A standard highway charging hub (10 HPC chargers × 150 kW) requires **1.5 MW** minimum — this is the `Moderate/Congested` boundary.
- A full-scale hub (60+ chargers) requires ~9 MW — rounded up to **10 MW** as the `Moderate/Sufficient` boundary.
- References: ACEA *Charging Infrastructure Barometer 2024*; MITMA *Plan de Recuperación para la Movilidad Sostenible 2023*.

> These thresholds must be reproduced verbatim in the Analytical Report under the `grid_status` classification section.

In [ ]:
# ── Threshold constants (adjust here if you revise for the report) ────────────
THRESHOLD_SUFFICIENT = 10.0   # MW — comfortably supports large HPC hub
THRESHOLD_MODERATE   =  1.5   # MW — minimum for a standard 10-charger hub

def classify_grid(mw):
    """Assign grid_status based on available capacity in MW."""
    if pd.isna(mw):
        return 'Moderate'   # Conservative default when capacity data is missing
    if mw >= THRESHOLD_SUFFICIENT:
        return 'Sufficient'
    elif mw >= THRESHOLD_MODERATE:
        return 'Moderate'
    else:
        return 'Congested'

if 'available_mw' in df.columns:
    df['grid_status'] = df['available_mw'].apply(classify_grid)
else:
    # Fallback: classify as Moderate if capacity data unavailable
    df['grid_status'] = 'Moderate'
    print('WARNING: available_mw column not found — all nodes classified as Moderate.')

status_counts = df['grid_status'].value_counts()
status_pct    = (status_counts / len(df) * 100).round(1)

print('GRID STATUS CLASSIFICATION')
print('-' * 45)
print(f'  Threshold Sufficient : ≥ {THRESHOLD_SUFFICIENT} MW')
print(f'  Threshold Moderate   : {THRESHOLD_MODERATE} – {THRESHOLD_SUFFICIENT} MW')
print(f'  Threshold Congested  : < {THRESHOLD_MODERATE} MW')
print()
for status in ['Sufficient', 'Moderate', 'Congested']:
    n   = status_counts.get(status, 0)
    pct = status_pct.get(status, 0.0)
    print(f'  {status:<12} : {n:>5,} nodes  ({pct:.1f}%)')
print(f'  {"TOTAL":<12} : {len(df):>5,} nodes')

## 6. Grid Status Map

Visual check that the classification logic produces a geographically coherent pattern. We expect:
- Green (Sufficient) nodes to cluster near major industrial zones and well-developed grid corridors
- Red (Congested) nodes to appear in rural and inland areas where grid investment has been lower
- Yellow (Moderate) to form a transitional band between the two

In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))

STATUS_COLORS = {'Sufficient': '#2ca02c', 'Moderate': '#ff7f0e', 'Congested': '#d62728'}
STATUS_SIZES  = {'Sufficient': 12, 'Moderate': 8, 'Congested': 6}

for status, color in STATUS_COLORS.items():
    sub = df[df['grid_status'] == status]
    ax.scatter(
        sub['longitude'], sub['latitude'],
        c=color, s=STATUS_SIZES[status],
        alpha=0.75, label=f'{status} ({len(sub):,})'
    )

# Overlay RTIG road network if available
if os.path.exists(RTIG_PATH):
    roads = gpd.read_file(RTIG_PATH)
    roads.plot(ax=ax, color='lightgrey', linewidth=0.4, alpha=0.5, zorder=0)
    ax.text(0.02, 0.02, 'Grey lines = RTIG interurban roads (M1 output)',
            transform=ax.transAxes, fontsize=7, color='grey')

ax.set_title(
    f'Endesa Grid Nodes — grid_status Classification\n'
    f'(Sufficient ≥{THRESHOLD_SUFFICIENT} MW | Moderate {THRESHOLD_MODERATE}–{THRESHOLD_SUFFICIENT} MW | Congested <{THRESHOLD_MODERATE} MW)',
    fontsize=11
)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_xlim(-10, 5)
ax.set_ylim(27, 45)
ax.legend(title='grid_status', fontsize=9, title_fontsize=9, loc='upper left')
plt.tight_layout()
plt.savefig('../data/interim/m4_endesa_grid_status_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grid status map saved.')

## 7. Nearest-Substation Lookup Function

The network optimisation notebook (M6) will call `get_grid_status(lat, lon)` to assign `grid_status` to each proposed charging location. This cell builds a spatial index (KD-tree) over the Endesa nodes and exports the lookup function.

**Spatial matching methodology (per datathon Rule 1):**  
For each proposed charging station, we identify the nearest Endesa substation by Euclidean distance on WGS84 coordinates. At Spain's latitude, 1° ≈ 80 km (latitude) and ≈ 65 km (longitude) — acceptable for substation matching since the search is always for the nearest node, not a fixed-radius buffer.

The matched substation's `available_mw` and `grid_status` are assigned to the proposed location.

In [ ]:
# ── Build KD-tree over clean Endesa nodes ─────────────────────────────────────
coords = df[['latitude', 'longitude']].values
tree   = cKDTree(coords)

def get_grid_status(lat, lon):
    """
    Return (grid_status, available_mw, nearest_node_dist_deg) for a given
    WGS84 coordinate by finding the nearest Endesa substation.

    Parameters
    ----------
    lat : float  Geographic latitude in decimal degrees
    lon : float  Geographic longitude in decimal degrees

    Returns
    -------
    dict with keys: grid_status, available_mw, nearest_node_id, dist_deg
    """
    dist, idx = tree.query([lat, lon])
    row = df.iloc[idx]
    return {
        'grid_status'    : row['grid_status'],
        'available_mw'   : row.get('available_mw', np.nan),
        'nearest_node_id': row.get('node_id', idx),
        'dist_deg'       : round(dist, 4),
        'distributor'    : 'Endesa'
    }

# ── Smoke test ────────────────────────────────────────────────────────────────
# A-4 near Córdoba (Endesa zone)
test_result = get_grid_status(37.88, -4.78)
print('Smoke test — A-4 near Córdoba:')
for k, v in test_result.items():
    print(f'  {k:<20} : {v}')

print()
print('get_grid_status() is ready for use by M6 (network optimisation).')

## 8. Export Clean Dataset

Export the cleaned and classified Endesa substation dataset to `data/interim/`. M6 will load this file to perform nearest-node matching for all proposed charging locations in Endesa's distribution zone.

In [ ]:
export_cols = [c for c in
    ['node_id', 'node_name', 'municipality', 'province',
     'latitude', 'longitude', 'available_mw', 'voltage_kv',
     'access_type', 'grid_status']
    if c in df.columns
]

df[export_cols].to_csv(OUT_PATH, index=False)
size_kb = os.path.getsize(OUT_PATH) / 1024

print(f'Saved   : {OUT_PATH}')
print(f'Size    : {size_kb:.1f} KB')
print(f'Records : {len(df):,}')

## 9. Output Verification

Per evaluation criterion T5, all output file structures must be printed and verified in the notebook.

In [ ]:
df_v = pd.read_csv(OUT_PATH)

print('OUTPUT VERIFICATION — m4_endesa_grid_capacity.csv')
print('-' * 60)
print(f'  File    : {OUT_PATH}')
print(f'  Rows    : {len(df_v):,}')
print(f'  Columns : {df_v.columns.tolist()}')
print()
print('Column dtypes:')
print(df_v.dtypes.to_string())
print()
print('Sample (first 3 rows):')
print(df_v.head(3).to_string(index=False))
print()
print('grid_status distribution:')
print(df_v['grid_status'].value_counts().to_string())
print()
print('Coordinate ranges:')
print(f'  Latitude  : {df_v["latitude"].min():.3f} → {df_v["latitude"].max():.3f}')
print(f'  Longitude : {df_v["longitude"].min():.3f} → {df_v["longitude"].max():.3f}')

# ── Assertions ────────────────────────────────────────────────────────────────
assert len(df_v) > 0, 'ERROR: Output file is empty'
assert 'grid_status' in df_v.columns, 'ERROR: grid_status column missing'
assert set(df_v['grid_status'].unique()).issubset({'Sufficient', 'Moderate', 'Congested'}), \
    'ERROR: Invalid grid_status values found'
assert df_v['latitude'].between(27, 45).all(), 'ERROR: Latitude out of Spain range'
assert df_v['longitude'].between(-20, 5).all(), 'ERROR: Longitude out of Spain range'

print()
print('✓ All assertions passed.')

## M4 Summary — Key Takeaways for the Analytical Report

| Finding | Value | Implication |
|---|---|---|
| Endesa distribution zone | Andalucía, Extremadura, Cataluña, Canarias, Baleares + parts of Aragón & Castilla-La Mancha | Covers A-4, A-7, A-92 and key southern corridors |
| Classification thresholds | Sufficient ≥10 MW \| Moderate 1.5–10 MW \| Congested <1.5 MW | Based on HPC hub sizing (10×150 kW = 1.5 MW minimum) |
| Lookup method | Nearest-neighbour KD-tree on WGS84 coords | Applied in M6 to classify every proposed charging location |
| Mandatory rule compliance | Rule 1: grid_status from distributor capacity data ✓ | Thresholds documented here and in Analytical Report |